In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import (
    EndpointType, VectorIndexType, DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn, PipelineType)

w = WorkspaceClient()

# Create the endpoint 
w.vector_search_endpoints.create_endpoint_and_wait(
    name="steam_vs", endpoint_type=EndpointType.STANDARD)

# Create the index
w.vector_search_indexes.create_index(
    name="steam_data.default.reviews_index",
    endpoint_name="steam_vs",
    primary_key="review_id",
    index_type=VectorIndexType.DELTA_SYNC,
    delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
        source_table="steam_data.default.reviews_rag",
        pipeline_type=PipelineType.TRIGGERED,
        embedding_source_columns=[EmbeddingSourceColumn(
            name="review_clean",
            embedding_model_endpoint_name="databricks-gte-large-en")])) 

In [0]:
ep = w.vector_search_endpoints.get_endpoint(endpoint_name="steam_vs")
print(ep.endpoint_status)

In [0]:
import time

for i in range(20):
    s = w.vector_search_indexes.get_index(index_name="steam_data.default.reviews_index").status
    print(i, "ready:", s.ready, "| rows:", s.indexed_row_count, "|", s.message)
    if s.ready:
        break
    time.sleep(60)

In [0]:
from databricks.sdk import WorkspaceClient
from mlflow.deployments import get_deploy_client

w = WorkspaceClient()
client = get_deploy_client("databricks")

In [0]:
s = w.vector_search_indexes.get_index(index_name="steam_data.default.reviews_index").status
print(s.ready, s.indexed_row_count)

In [0]:
%sql
SELECT COUNT(*) FROM steam_data.default.reviews_rag;

In [0]:
def show(query, k=6, app_id=None):
    res = w.vector_search_indexes.query_index(
        index_name="steam_data.default.reviews_index",
        columns=["review_id", "app_id", "name", "review_clean"],
        query_text=query, num_results=k,
        filters_json=('{"app_id": %d}' % app_id) if app_id else None)
    for r in res.result.data_array:
        print(r[0], "|", r[2], "|", r[3][:130])

# show("bugs and crashes")
# show("too expensive, not worth the price")
show("PUBG")

In [0]:
def ask(question, app_id=None, k=8, model="databricks-meta-llama-3-3-70b-instruct"):
    res = w.vector_search_indexes.query_index(
        index_name="steam_data.default.reviews_index",
        columns=["review_id", "app_id", "name", "review_clean"],
        query_text=question, num_results=k,
        filters_json=('{"app_id": %d}' % app_id) if app_id else None)
    rows = res.result.data_array
    context = "\n".join(f"[{r[0]}] ({r[2]}) {r[3]}" for r in rows)
    prompt = f"""Answer using ONLY the reviews below. Cite review IDs in brackets.
If the reviews do not answer the question, say so. Do not use outside knowledge. 

Question: {question}

Reviews:
{context}"""
    resp = client.predict(endpoint=model, inputs={
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 400, "temperature": 0})
    return resp["choices"][0]["message"]["content"], rows

answer, evidence = ask("Do players think this game is worth the price?")
print(answer)

In [0]:
%sql
SELECT app_id, name, COUNT(*) AS n
FROM steam_data.default.reviews_rag
GROUP BY app_id, name
ORDER BY n DESC
LIMIT 10;

In [0]:
df=spark.table("steam_data.default.reviews_rag")
print(df[])